# Phase 2 — LSE Computation

For each of the 60 meta-training datasets, runs all 6 pseudo-label generation methods,
applies Hungarian alignment, trains a fixed RF, and records LSE.

**Output**: `data/meta_table/meta_training.csv`  
Columns: `dataset_id | LSE_kmeans | LSE_dbscan | LSE_agg | LSE_gmm | LSE_autoenc | LSE_dictlearn | best_method`

**Rules (from CLAUDE.md)**:
- Random Forest: default sklearn params, `random_state=42`, never tuned
- `random_state=42` for all sklearn objects, `torch.manual_seed(42)` for PyTorch
- True labels seen only during Hungarian alignment and final evaluation
- Showcase datasets never in this table

In [1]:
import os, time, warnings
from pathlib import Path
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import openml

from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.cluster import KMeans, DBSCAN, AgglomerativeClustering
from sklearn.mixture import GaussianMixture
from sklearn.decomposition import DictionaryLearning
from sklearn.neighbors import NearestNeighbors

import torch
import torch.nn as nn

# ── Paths ─────────────────────────────────────────────────────────────────────
def find_project_root(start=None):
    cur = Path(start or os.getcwd()).resolve()
    for path in (cur, *cur.parents):
        if (path / 'CLAUDE.md').exists() and (path / 'src').exists():
            return str(path)
    raise RuntimeError('Project root not found; run from the repo root or notebooks folder')

ROOT       = find_project_root()
RAW_DIR    = os.path.join(ROOT, 'data', 'raw')
META_DIR   = os.path.join(ROOT, 'data', 'meta_table')
MANIFEST   = os.path.join(META_DIR, 'dataset_manifest.csv')
CHECKPOINT = os.path.join(META_DIR, 'lse_checkpoint.csv')
OUTPUT     = os.path.join(META_DIR, 'meta_training.csv')

openml.config.cache_directory = RAW_DIR

# ── Global seeds ──────────────────────────────────────────────────────────────
SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

print('Paths OK')

Paths OK


In [2]:
# ── Load manifest + showcase exclusion guard ──────────────────────────────────
SHOWCASE_IDS = {61, 187, 15, 53, 40966, 37, 54, 1590, 1597}

manifest = pd.read_csv(MANIFEST)
leaked = set(manifest['dataset_id']) & SHOWCASE_IDS
assert len(leaked) == 0, f'Showcase leak in manifest: {leaked}'

print(f'Manifest: {len(manifest)} datasets')
manifest.head()

Manifest: 60 datasets


,dataset_id,name,n_instances,n_features,n_classes
0,43924,eucalyptus,736,19,5
1,732,fri_c0_250_50,250,50,2
2,1459,artificial-characters,10218,7,10
3,1495,qualitative-bankruptcy,250,6,2
4,4538,GesturePhaseSegmentationProcessed,9873,32,5


## Helper functions

In [3]:
# ── Preprocessing ─────────────────────────────────────────────────────────────

def load_and_split(dataset_id):
    """Download (cached), encode labels, 80/20 stratified split."""
    ds = openml.datasets.get_dataset(
        dataset_id,
        download_data=True,
        download_qualities=False,
        download_features_meta_data=False,
    )
    X, y, _, _ = ds.get_data(
        dataset_format='dataframe',
        target=ds.default_target_attribute,
    )
    # Drop any remaining non-numeric cols; coerce to float
    X = X.select_dtypes(include=[np.number]).astype(float)
    le = LabelEncoder()
    y_enc = le.fit_transform(y.astype(str))

    X_tr, X_te, y_tr, y_te = train_test_split(
        X.values, y_enc,
        test_size=0.2,
        random_state=SEED,
        stratify=y_enc,
    )
    return X_tr, X_te, y_tr, y_te


def scale(X_tr, X_te):
    """Fit StandardScaler on train, apply to both."""
    sc = StandardScaler()
    return sc.fit_transform(X_tr), sc.transform(X_te)

In [4]:
# ── LSE computation ───────────────────────────────────────────────────────────

def compute_lse(X_tr_raw, y_tr, X_te_raw, y_te, pseudo_labels_tr, gt_acc,
                method_name='', dataset_name='', verbose=True):
    """
    Compute LSE with full diagnostic logging.

    Returns
    -------
    lse : float
        The LSE ratio (acc / gt_acc), unclipped.
    diag : dict
        Diagnostic information about this run, one flat dict per (dataset, method).
    """
    diag = {'dataset': dataset_name, 'method': method_name}

    # --- 1. Cluster structure diagnostics ---
    unique_clusters, cluster_sizes = np.unique(pseudo_labels_tr, return_counts=True)
    n_clusters = len(unique_clusters)
    n_classes  = len(np.unique(y_tr))
    largest_cluster_frac = cluster_sizes.max() / cluster_sizes.sum()

    diag['n_clusters']           = int(n_clusters)
    diag['n_classes']            = int(n_classes)
    diag['largest_cluster_frac'] = round(float(largest_cluster_frac), 3)
    diag['cluster_collapse']     = bool(n_clusters < n_classes)
    diag['cluster_degenerate']   = bool(largest_cluster_frac > 0.90)

    # --- 2. Mapping diagnostics (majority vote per cluster) ---
    mapping = {}
    for cid in unique_clusters:
        mask = pseudo_labels_tr == cid
        mapping[int(cid)] = int(np.bincount(y_tr[mask]).argmax())

    unique_mapped_classes = len(set(mapping.values()))
    diag['unique_mapped_classes'] = int(unique_mapped_classes)
    diag['mapping_collapse']      = bool(unique_mapped_classes < n_classes)

    # --- 3. Train RF on raw cluster IDs ---
    rf = RandomForestClassifier(random_state=SEED)
    rf.fit(X_tr_raw, pseudo_labels_tr)
    train_acc_on_clusters = rf.score(X_tr_raw, pseudo_labels_tr)
    diag['rf_train_acc_on_clusters'] = round(float(train_acc_on_clusters), 3)
    diag['rf_underfit_clusters']     = bool(train_acc_on_clusters < 0.80)

    # --- 4. Predict on test set and translate via mapping ---
    test_preds_ids = rf.predict(X_te_raw)
    fallback = int(np.bincount(y_tr).argmax())
    test_preds_classes = np.array(
        [mapping.get(int(c), fallback) for c in test_preds_ids]
    )
    diag['unique_predicted_classes'] = int(len(np.unique(test_preds_classes)))

    # --- 5. Accuracy breakdown ---
    acc = float((test_preds_classes == y_te).mean())
    majority_baseline = float((y_te == fallback).mean())
    random_baseline   = 1.0 / n_classes

    diag['acc']               = round(acc, 3)
    diag['majority_baseline'] = round(majority_baseline, 3)
    diag['random_baseline']   = round(random_baseline, 3)
    diag['gt_acc']            = round(float(gt_acc), 3)

    # --- 6. Two versions of LSE ---
    lse_ratio = acc / gt_acc if gt_acc > 0 else 0.0
    lift_denom = gt_acc - majority_baseline
    lse_lift = (acc - majority_baseline) / lift_denom if lift_denom > 1e-6 else 0.0
    diag['lse_ratio'] = round(float(lse_ratio), 3)
    diag['lse_lift']  = round(float(lse_lift), 3)

    # --- 7. Failure-mode flags ---
    diag['beats_majority']    = bool(acc > majority_baseline + 0.01)
    diag['matches_majority']  = bool(abs(acc - majority_baseline) <= 0.01)
    diag['gt_beats_majority'] = bool(gt_acc > majority_baseline + 0.05)

    # --- 8. Verbose per-method print ---
    if verbose:
        flags = []
        if diag['cluster_collapse']:      flags.append('CLUSTER_COLLAPSE')
        if diag['cluster_degenerate']:    flags.append('DEGENERATE_CLUSTER')
        if diag['mapping_collapse']:      flags.append('MAPPING_COLLAPSE')
        if diag['matches_majority']:      flags.append('MATCHES_MAJORITY')
        if diag['rf_underfit_clusters']:  flags.append('RF_UNDERFIT')
        if not diag['gt_beats_majority']: flags.append('GT_NEAR_MAJORITY')
        flag_str = '  [' + ', '.join(flags) + ']' if flags else ''
        print(f"    {method_name:10s}  n_cl={n_clusters}/{n_classes}  "
              f"maj_base={majority_baseline:.3f}  acc={acc:.3f}  "
              f"lse_r={lse_ratio:.3f}  lse_lift={lse_lift:.3f}{flag_str}")

    return float(lse_ratio), diag


def groundtruth_accuracy(X_tr_raw, y_tr, X_te_raw, y_te):
    """RF trained on true labels — denominator for all LSE values."""
    rf = RandomForestClassifier(random_state=SEED)
    rf.fit(X_tr_raw, y_tr)
    return rf.score(X_te_raw, y_te)

## Clustering methods

In [5]:
# ── 1. k-means ────────────────────────────────────────────────────────────────

def pseudo_kmeans(X_scaled, n_classes):
    km = KMeans(n_clusters=n_classes, random_state=SEED, n_init=10)
    return km.fit_predict(X_scaled)

In [6]:
# ── 2. DBSCAN (auto-tune eps, noise reassignment) ─────────────────────────────

def pseudo_dbscan(X_scaled, n_classes):
    k = 5
    nbrs = NearestNeighbors(n_neighbors=k).fit(X_scaled)
    dists, _ = nbrs.kneighbors(X_scaled)
    knn_dists = dists[:, -1]

    labels = None
    for pct in [90, 75, 60, 45, 30]:
        eps = float(np.percentile(knn_dists, pct))
        if eps <= 0:
            continue
        db = DBSCAN(eps=eps, min_samples=k).fit(X_scaled)
        n_clusters = len(set(db.labels_) - {-1})
        if n_clusters >= 2:
            labels = db.labels_.copy()
            break

    # Fallback: no valid eps found
    if labels is None or len(set(labels) - {-1}) < 2:
        return pseudo_kmeans(X_scaled, n_classes)

    # Reassign noise points to nearest non-noise neighbour's cluster
    noise_mask = labels == -1
    if noise_mask.any():
        non_noise_idx = np.where(~noise_mask)[0]
        nn1 = NearestNeighbors(n_neighbors=1).fit(X_scaled[~noise_mask])
        _, idx = nn1.kneighbors(X_scaled[noise_mask])
        labels[noise_mask] = labels[non_noise_idx[idx.ravel()]]

    return labels

In [7]:
# ── 3. Agglomerative (Ward linkage) ──────────────────────────────────────────

def pseudo_agglomerative(X_scaled, n_classes):
    agg = AgglomerativeClustering(n_clusters=n_classes, linkage='ward')
    return agg.fit_predict(X_scaled)

In [8]:
# ── 4. GMM ────────────────────────────────────────────────────────────────────

def pseudo_gmm(X_scaled, n_classes):
    gmm = GaussianMixture(
        n_components=n_classes,
        random_state=SEED,
        max_iter=200,
        reg_covar=1e-5,   # numerical stability
    )
    gmm.fit(X_scaled)
    return gmm.predict(X_scaled)

In [9]:
# ── 5. Autoencoder + k-means ──────────────────────────────────────────────────

class _Autoencoder(nn.Module):
    def __init__(self, input_dim, bottleneck_dim):
        super().__init__()
        h = max(64, input_dim * 2)
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, h), nn.ReLU(),
            nn.Linear(h, bottleneck_dim),
        )
        self.decoder = nn.Sequential(
            nn.Linear(bottleneck_dim, h), nn.ReLU(),
            nn.Linear(h, input_dim),
        )

    def forward(self, x):
        z = self.encoder(x)
        return self.decoder(z), z


def pseudo_autoencoder(X_scaled, n_classes, n_epochs=100, batch_size=256):
    torch.manual_seed(SEED)
    input_dim      = X_scaled.shape[1]
    bottleneck_dim = max(n_classes * 2, 8)

    # Sub-sample for training if dataset is large
    MAX_AE_SAMPLES = 8000
    if len(X_scaled) > MAX_AE_SAMPLES:
        rng   = np.random.default_rng(SEED)
        idx   = rng.choice(len(X_scaled), MAX_AE_SAMPLES, replace=False)
        X_fit = X_scaled[idx]
    else:
        X_fit = X_scaled

    tensor_fit = torch.tensor(X_fit, dtype=torch.float32)
    model      = _Autoencoder(input_dim, bottleneck_dim)
    optim      = torch.optim.Adam(model.parameters(), lr=1e-3)
    loss_fn    = nn.MSELoss()

    model.train()
    for _ in range(n_epochs):
        perm = torch.randperm(len(tensor_fit))
        for i in range(0, len(tensor_fit), batch_size):
            batch = tensor_fit[perm[i:i + batch_size]]
            recon, _ = model(batch)
            loss = loss_fn(recon, batch)
            optim.zero_grad()
            loss.backward()
            optim.step()

    # Encode ALL training samples
    model.eval()
    with torch.no_grad():
        _, embeddings = model(torch.tensor(X_scaled, dtype=torch.float32))
    embeddings = embeddings.numpy()

    return pseudo_kmeans(embeddings, n_classes)

In [10]:
# ── 6. Dictionary Learning ────────────────────────────────────────────────────

def pseudo_dictlearn(X_scaled, n_classes):
    n_components = max(n_classes, 2)

    # Sub-sample for fitting if dataset is large (DL is O(n^2))
    MAX_DL_SAMPLES = 3000
    if len(X_scaled) > MAX_DL_SAMPLES:
        rng   = np.random.default_rng(SEED)
        idx   = rng.choice(len(X_scaled), MAX_DL_SAMPLES, replace=False)
        X_fit = X_scaled[idx]
    else:
        X_fit = X_scaled

    dl = DictionaryLearning(
        n_components=n_components,
        max_iter=200,
        random_state=SEED,
        transform_algorithm='lasso_lars',
        n_jobs=1,
    )
    dl.fit(X_fit)

    # Transform ALL training samples
    codes = dl.transform(X_scaled)          # shape (n_train, n_components)
    labels = np.argmax(np.abs(codes), axis=1)
    return labels

## Main loop with checkpointing

In [11]:
METHODS = {
    'LSE_kmeans'   : pseudo_kmeans,
    'LSE_dbscan'   : pseudo_dbscan,
    'LSE_agg'      : pseudo_agglomerative,
    'LSE_gmm'      : pseudo_gmm,
    'LSE_autoenc'  : pseudo_autoencoder,
    'LSE_dictlearn': pseudo_dictlearn,
}

# ── Resume from checkpoint if it exists ──────────────────────────────────────
if os.path.exists(CHECKPOINT):
    done_df  = pd.read_csv(CHECKPOINT)
    done_ids = set(done_df['dataset_id'])
    results  = done_df.to_dict('records')
    print(f'Resuming — {len(done_ids)} datasets already processed')
else:
    done_ids = set()
    results  = []
    print('Starting fresh')

all_diagnostics = []  # Collected only for datasets processed in THIS run.
                       # Resumed checkpoints won't re-log. To get diagnostics for
                       # all 60 datasets, delete lse_checkpoint.csv before running.

# ── Main loop ─────────────────────────────────────────────────────────────────
total = len(manifest)

for i, row in manifest.iterrows():
    did   = int(row['dataset_id'])
    name  = row['name']
    n_cls = int(row['n_classes'])

    if did in done_ids:
        continue

    t0  = time.time()
    rec = {'dataset_id': did}

    try:
        X_tr, X_te, y_tr, y_te = load_and_split(did)

        if X_tr.shape[1] == 0:
            raise ValueError('No numeric features after loading')

        X_tr_sc, X_te_sc = scale(X_tr, X_te)

        gt_acc = groundtruth_accuracy(X_tr, y_tr, X_te, y_te)

        if gt_acc == 0:
            raise ValueError(f'GT accuracy is 0 — skipping dataset {did}')

        print(f'[{i+1:3d}/{total}] {name[:35]:35s}  gt={gt_acc:.3f}  n_cls={n_cls}')

        for col, fn in METHODS.items():
            rec[col] = float('nan')
            try:
                pseudo = fn(X_tr_sc, n_cls)
                lse, diag = compute_lse(
                    X_tr, y_tr, X_te, y_te, pseudo, gt_acc,
                    method_name=col.replace('LSE_', ''),
                    dataset_name=name,
                    verbose=True,
                )
                rec[col] = round(lse, 4)
                all_diagnostics.append(diag)
            except Exception as e:
                print(f'    {col} FAILED: {e}')
                all_diagnostics.append({
                    'dataset': name,
                    'method': col.replace('LSE_', ''),
                    'failed': True,
                    'error': str(e)[:200],
                })

        elapsed = time.time() - t0
        best    = max((col for col in METHODS if not np.isnan(rec.get(col, np.nan))),
                      key=lambda c: rec.get(c, -1))
        rec['best_method'] = best.replace('LSE_', '')
        rec['gt_accuracy'] = round(gt_acc, 4)

        results.append(rec)
        done_ids.add(did)
        pd.DataFrame(results).to_csv(CHECKPOINT, index=False)

        print(f'           best={rec["best_method"]}  ({elapsed:.1f}s)')

    except Exception as e:
        print(f'[{i+1:3d}/{total}] FAIL  id={did}  {name}  — {e}')
        rec.update({c: float('nan') for c in METHODS})
        rec['best_method'] = 'FAILED'
        rec['gt_accuracy'] = float('nan')
        results.append(rec)
        done_ids.add(did)
        pd.DataFrame(results).to_csv(CHECKPOINT, index=False)

print(f'\nDone. {len(results)} rows collected.')

# ── Inline diagnostics summary ────────────────────────────────────────────────
if all_diagnostics:
    _d = pd.DataFrame(all_diagnostics)
    print('\n=== Failure-mode frequency ===')
    for flag in ['cluster_collapse', 'cluster_degenerate', 'mapping_collapse',
                 'matches_majority', 'rf_underfit_clusters']:
        if flag in _d.columns:
            n   = int(_d[flag].fillna(False).sum())
            pct = _d[flag].fillna(False).mean() * 100
            print(f'  {flag:25s}  {pct:5.1f}%  ({n} runs)')

    if 'matches_majority' in _d.columns:
        print('\n=== Datasets where EVERY method matches majority ===')
        bad      = _d.groupby('dataset')['matches_majority'].all()
        bad_list = bad[bad].index.tolist()
        print('  ' + ', '.join(bad_list) if bad_list else '  (none)')

    if 'lse_ratio' in _d.columns:
        print('\n=== Per-dataset LSE std across methods ===')
        print(_d.groupby('dataset')['lse_ratio'].std().describe().round(3).to_string())

Starting fresh
[  1/60] eucalyptus                           gt=0.439  n_cls=5
    kmeans      n_cl=5/5  maj_base=0.291  acc=0.345  lse_r=0.785  lse_lift=0.364  [MAPPING_COLLAPSE]
    dbscan      n_cl=5/5  maj_base=0.291  acc=0.345  lse_r=0.785  lse_lift=0.364  [MAPPING_COLLAPSE]
    agg         n_cl=5/5  maj_base=0.291  acc=0.345  lse_r=0.785  lse_lift=0.364  [MAPPING_COLLAPSE]
    gmm         n_cl=5/5  maj_base=0.291  acc=0.345  lse_r=0.785  lse_lift=0.364  [MAPPING_COLLAPSE]
    autoenc     n_cl=5/5  maj_base=0.291  acc=0.345  lse_r=0.785  lse_lift=0.364  [MAPPING_COLLAPSE]
    dictlearn   n_cl=5/5  maj_base=0.291  acc=0.392  lse_r=0.892  lse_lift=0.682  [MAPPING_COLLAPSE]
           best=dictlearn  (7.0s)
[  2/60] fri_c0_250_50                        gt=0.820  n_cls=2
    kmeans      n_cl=2/2  maj_base=0.540  acc=0.540  lse_r=0.659  lse_lift=0.000  [MAPPING_COLLAPSE, MATCHES_MAJORITY]
    dbscan      n_cl=2/2  maj_base=0.540  acc=0.540  lse_r=0.659  lse_lift=0.000  [MAPPING_COLLAPS

## Build `meta_training.csv`

In [ ]:
# ── Assemble final table ──────────────────────────────────────────────────────
df = pd.DataFrame(results)

# Drop datasets where ALL methods failed
lse_cols = list(METHODS.keys())
all_nan  = df[lse_cols].isna().all(axis=1)
if all_nan.any():
    print(f'Dropping {all_nan.sum()} fully-failed datasets: {df.loc[all_nan, "dataset_id"].tolist()}')
    df = df[~all_nan].reset_index(drop=True)

# Recompute best_method from LSE columns (robust to partial failures)
def _best(row):
    vals = {c: row[c] for c in lse_cols if not np.isnan(row[c])}
    if not vals:
        return 'FAILED'
    return max(vals, key=vals.get).replace('LSE_', '')

df['best_method'] = df.apply(_best, axis=1)

# Column order
col_order = ['dataset_id'] + lse_cols + ['best_method', 'gt_accuracy']
df = df[col_order]

# Showcase exclusion final check
leaked = set(df['dataset_id']) & SHOWCASE_IDS
assert len(leaked) == 0, f'Showcase leak: {leaked}'

df.to_csv(OUTPUT, index=False)
print(f'Saved → {OUTPUT}')
print(f'Shape : {df.shape}')
df

In [ ]:
# ── Write diagnostics CSV and print failure-mode summary ──────────────────────

DIAG_OUTPUT = os.path.join(META_DIR, 'diagnostics.csv')

if len(all_diagnostics) == 0:
    print('No diagnostics collected (likely all datasets resumed from checkpoint).')
    print('To collect diagnostics, delete lse_checkpoint.csv and rerun.')
else:
    diag_df = pd.DataFrame(all_diagnostics)
    diag_df.to_csv(DIAG_OUTPUT, index=False)
    print(f'Saved → {DIAG_OUTPUT}')
    print(f'Shape : {diag_df.shape}')

    print('\n=== Failure-mode frequency across all runs ===')
    flag_cols = ['cluster_collapse', 'cluster_degenerate', 'mapping_collapse',
                 'matches_majority', 'rf_underfit_clusters']
    for flag in flag_cols:
        if flag in diag_df.columns:
            pct = diag_df[flag].fillna(False).mean() * 100
            print(f'  {flag:25s}  {pct:5.1f}%  ({int(diag_df[flag].fillna(False).sum())} runs)')

    if 'matches_majority' in diag_df.columns:
        print('\n=== Datasets where EVERY method matches majority (unclusterable) ===')
        bad = diag_df.groupby('dataset')['matches_majority'].all()
        bad_datasets = bad[bad].index.tolist()
        if bad_datasets:
            for d in bad_datasets:
                print(f'  {d}')
        else:
            print('  (none — good)')

    if 'lse_lift' in diag_df.columns:
        print('\n=== Top 15 datasets by max lse_lift across methods ===')
        top_lift = diag_df.groupby('dataset')['lse_lift'].max().sort_values(ascending=False).head(15)
        for d, v in top_lift.items():
            print(f'  {d:40s}  lse_lift={v:.3f}')

    if 'lse_ratio' in diag_df.columns:
        print('\n=== Per-dataset LSE variance across methods (meta-learnability signal) ===')
        variance = diag_df.groupby('dataset')['lse_ratio'].std().describe()
        print(variance.round(3).to_string())
        print('\n(If mean std is < 0.05, methods are too similar for a meta-learner to distinguish.)')

In [14]:
# ── Validation & summary ──────────────────────────────────────────────────────

print('=== LSE descriptive statistics ===')
print(df[lse_cols].describe().round(3).to_string())

print('\n=== Best-method distribution ===')
print(df['best_method'].value_counts())

print('\n=== NaN counts per method ===')
print(df[lse_cols].isna().sum())

# Sanity: LSE values should be in [0, ~1.5]
for col in lse_cols:
    valid = df[col].dropna()
    assert (valid >= 0).all(), f'{col} has negative LSE'
    assert (valid <= 1.5).all(), f'{col} has LSE > 1.5'

print('\nAll sanity checks passed.')
print(f'meta_training.csv ready for Phase 3 (meta-feature extraction).')

=== LSE descriptive statistics ===
       LSE_kmeans  LSE_dbscan  LSE_agg  LSE_gmm  LSE_autoenc  LSE_dictlearn
count      51.000      51.000   41.000   51.000       51.000         51.000
mean        0.767       0.750    0.761    0.780        0.773          0.748
std         0.182       0.218    0.196    0.181        0.185          0.205
min         0.247       0.188    0.242    0.253        0.276          0.258
25%         0.654       0.612    0.631    0.659        0.661          0.628
50%         0.737       0.769    0.771    0.801        0.764          0.725
75%         0.946       0.952    0.952    0.928        0.950          0.933
max         1.062       1.073    1.062    1.062        1.062          1.111

=== Best-method distribution ===
best_method
dbscan       17
kmeans       14
gmm           6
dictlearn     5
autoenc       5
agg           4
Name: count, dtype: int64

=== NaN counts per method ===
LSE_kmeans        0
LSE_dbscan        0
LSE_agg          10
LSE_gmm           0
LS